## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

groq_api_key=os.getenv("GROQ_API_KEY")

In [4]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

model=ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", groq_api_key=groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002B3AEEA3DA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002B3AEF78A70>, model_name='meta-llama/llama-4-scout-17b-16e-instruct', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from langchain_core.messages import HumanMessage, AIMessage
model.invoke([HumanMessage(content="Hi, my name is Matheus!")])

AIMessage(content="Nice to meet you, Matheus! How's your day going so far?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 18, 'total_tokens': 35, 'completion_time': 0.038634329, 'completion_tokens_details': None, 'prompt_time': 0.000100208, 'prompt_tokens_details': None, 'queue_time': 0.15085854, 'total_time': 0.038734537}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_58fbeab661', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c49f4-00e7-7940-96c5-06dabb0c0f62-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 17, 'total_tokens': 35})

In [ ]:
# Aqui estamos adicionando "memoria" para a llm. Uma simples ideia!
model.invoke(
    [
        HumanMessage(content="WHi, my name is Matheus! I`m a data scientist."),
        AIMessage(content="Nice to meet you, Matheus! How's your day going so far?"),
        HumanMessage(content="Hey! What is my name and what do I do?")
    ]
)

AIMessage(content="According to our conversation, your name is **Matheus** and you're a **data scientist**!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 64, 'total_tokens': 85, 'completion_time': 0.04865909, 'completion_tokens_details': None, 'prompt_time': 0.000417853, 'prompt_tokens_details': None, 'queue_time': 0.151563993, 'total_time': 0.049076943}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_58fbeab661', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c49f5-d34b-7c02-9154-629119653ce7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 64, 'output_tokens': 21, 'total_tokens': 85})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [10]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [7]:
config = {"configurable": {"session_id": "chat1"}}

In [12]:
response = with_message_history.invoke(
    [
        HumanMessage(content="Hi, my name is Matheus! I`m a data scientist. I'm studying langchain!"),
    ],
    config=config
)

In [13]:
response.content

"Hello Matheus! Nice to meet you again! As a data scientist studying LangChain, you must be interested in exploring the intersection of natural language processing, knowledge graphs, and machine learning.\n\nLangChain is a powerful framework that enables the creation of applications that can understand, reason, and generate human-like text. It's built on top of large language models and allows developers to create custom applications that can interact with users in a more natural way.\n\nWhat aspects of LangChain are you most interested in, Matheus? Are you looking into:\n\n1. **Applications**: Building conversational interfaces, chatbots, or virtual assistants?\n2. **Knowledge Graphs**: Integrating domain-specific knowledge into your LangChain applications?\n3. **Reasoning**: Exploring the capabilities of LangChain's reasoning engine and how it can be used for decision-making?\n4. **Implementation**: Diving into the technical details of LangChain and implementing it for a specific use

In [14]:
with_message_history.invoke(
    [
        HumanMessage(content="What is my name?"),
    ],
    config=config
)

AIMessage(content="Your name is Matheus, and you're a data scientist studying LangChain!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 382, 'total_tokens': 398, 'completion_time': 0.036346682, 'completion_tokens_details': None, 'prompt_time': 0.011313098, 'prompt_tokens_details': None, 'queue_time': 0.150306945, 'total_time': 0.04765978}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_58fbeab661', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c4a04-4873-72d3-a7e1-5a50272a998a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 382, 'output_tokens': 16, 'total_tokens': 398})

In [ ]:
## change the config --> session id
config1 = {"configurable": {"session_id": "chat1"}}
response = with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config1 
)
print(f"Response content: {response.content}")
print()
config1 = {"configurable": {"session_id": "chat2"}} # Mudamos o session id!!!!
response = with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config1 
)
print(f"Response content: {response.content}")

Response content: Your name is Matheus, and you're a data scientist studying LangChain!

Response content: I don't have access to personal information, so I don't know your name. I'm here to provide general information and answer questions, though! Is there something specific you'd like to know or discuss?


In [19]:
response = with_message_history.invoke(
    [HumanMessage(content="Hey! My name is John!")],
    config=config1 
)
print(response.content)

response = with_message_history.invoke(
    [HumanMessage(content="Hey! What is my name?")],
    config=config1 
)
print(response.content)

Nice to meet you, John! I'm glad we got to have a little chat. If you want to talk about something specific or just shoot the breeze, I'm here to listen and help out. How's your day going so far?
Your name is John! We already established that earlier! How's it going, John?


In [ ]:
store

{'chat1': InMemoryChatMessageHistory(messages=[HumanMessage(content="Hi, my name is Matheus! I`m a data scientist. I'm studying langchain!", additional_kwargs={}, response_metadata={}), AIMessage(content="Nice to meet you, Matheus! LangChain is a fascinating topic, and I'm happy to help you with any questions or discussions you'd like to have about it.\n\nLangChain is a relatively new concept that combines language models, knowledge graphs, and reasoning to enable more advanced and interpretable natural language processing. It's an exciting area of research with many potential applications.\n\nWhat specifically are you studying about LangChain? Are you looking into its architecture, applications, or perhaps implementing it for a particular use case? I'm here to help and chat!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 30, 'total_tokens': 138, 'completion_time': 0.252364164, 'completion_tokens_details': None, 'prompt_time': 0.00

In [27]:
for d in store:
    print(f"Session id: {d}")
    print(store[d].messages)
    print()

Session id: chat1
[HumanMessage(content="Hi, my name is Matheus! I`m a data scientist. I'm studying langchain!", additional_kwargs={}, response_metadata={}), AIMessage(content="Nice to meet you, Matheus! LangChain is a fascinating topic, and I'm happy to help you with any questions or discussions you'd like to have about it.\n\nLangChain is a relatively new concept that combines language models, knowledge graphs, and reasoning to enable more advanced and interpretable natural language processing. It's an exciting area of research with many potential applications.\n\nWhat specifically are you studying about LangChain? Are you looking into its architecture, applications, or perhaps implementing it for a particular use case? I'm here to help and chat!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 30, 'total_tokens': 138, 'completion_time': 0.252364164, 'completion_tokens_details': None, 'prompt_time': 0.000746007, 'prompt_tokens_deta

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [8]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer all the question to the best of your ability."),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [9]:
chain.invoke({"messages": [HumanMessage(content="Hi, my name is Matheus!")]})

AIMessage(content="Nice to meet you, Matheus! How's your day going so far? Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 40, 'total_tokens': 72, 'completion_time': 0.072429245, 'completion_tokens_details': None, 'prompt_time': 0.000965932, 'prompt_tokens_details': None, 'queue_time': 0.213803866, 'total_time': 0.073395177}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_58fbeab661', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c4eed-608a-7932-884e-3faadf954a17-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 32, 'total_tokens': 72})

In [11]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [16]:
config = {"configurable": {"session_id": "chat3"}}
response = with_message_history.invoke(
    [
        HumanMessage(content="Hi, my name is Matheus!")
    ],
    config=config
)
response.content

'Matheus, I\'ve got it now! I\'ve seen your name three times, so I\'m sure to remember it. Feel free to share what\'s on your mind, and I\'ll do my best to help or chat with you. No need to reintroduce yourself, though - I\'ve got it saved in my " conversational memory" now!'

In [24]:
## Add more complexity.

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer all the question to the best of your ability in {language}."),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [27]:
response = chain.invoke(
    {
        "messages": [HumanMessage(content="Hi, my name is Matheus!")],
        "language": "Portuguese"
    }
)
response.content

'Olá Matheus! É um prazer conhecê-lo! Como posso ajudar você hoje?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [28]:
with_message_history = RunnableWithMessageHistory(
    chain, 
    get_session_history,
    input_messages_key="messages"
)

In [31]:
config = {"configurable": {"session_id": "chat4"}}

response = with_message_history.invoke(
    {"messages": [HumanMessage(content="Hi, this is Matheus!")], "language": "Portuguese"},
    config=config
)
response.content

'Olá Matheus! É um prazer conhecê-lo! Como posso ajudar você hoje?'

In [32]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="What is my name?")], "language": "Portuguese"},
    config=config
)
response.content

'Seu nome é Matheus!'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
`trim_messages` helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [40]:
from langchain_core.messages import SystemMessage, AIMessage, trim_messages

trimmer = trim_messages(
    max_tokens=45, 
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

Perceba que as mensagens:
```
HumanMessage(content="hi! I'm bob"),
AIMessage(content="hi!"),
```
sumiram, isto é por que o trimmer atuou aqui.

In [42]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
)

response = chain.invoke(
    {
    "messages": messages + [HumanMessage(content="What is my favourite ice cream flavor?")],
    "language": "English"
    }
)
response.content

"I'm not sure, I don't have that information. Would you like to share it with me?"

O trimmer atuou e nao conseguiu "capturar" o sabor favorito do sorvete.

In [43]:
response = chain.invoke(
    {
    "messages": messages + [HumanMessage(content="What math proble did I askt?")],
    "language": "English"
    }
)
response.content

'You asked me to solve 2 + 2.'

In [44]:
## Lets wrap this in the Message History


with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

config = {"configurable": {"session_id": "chat5"}}

In [48]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="What is my name?")], 
        "language": "Portuguese"
        },
    config=config
)
response.content

"I don't know your name. You didn't tell me."

In [49]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="What math problem did I ask?")], 
        "language": "English"
        },
    config=config
)
response.content

"You didn't ask a math problem. This conversation just started with your question about your name. What would you like to talk about or what math problem do you have? I'm here to help!"